# Week 1, Lab 1 — Intro & First Call to a Local Model

**Course:** Agentic AI Engineering — Local Models Edition
**Author:** Abhishek

Runs identically on **Google Colab** (Hugging Face `transformers`) and on
**your own PC** (Ollama) — no API key, no cost, either way.

## What you'll do
1. Detect whether you're on Colab or local, and pick the right backend
2. Make your first call to a local LLM
3. Try zero-shot vs. few-shot vs. chain-of-thought prompting


## 1. Environment detection

Run this cell as-is. It figures out where you are and tells you what to do next.


In [2]:
import importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BACKEND = "huggingface" if IN_COLAB else "ollama"

print(f"Detected environment: {'Google Colab' if IN_COLAB else 'Local PC'}")
print(f"Backend selected: {BACKEND}")

if IN_COLAB:
    print("\nTip: Runtime -> Change runtime type -> T4 GPU, for faster generation.")
else:
    print("\nMake sure Ollama is running: `ollama serve` in a terminal, and you've")
    print("pulled a model: `ollama pull llama3.2:3b`")


Detected environment: Google Colab
Backend selected: huggingface

Tip: Runtime -> Change runtime type -> T4 GPU, for faster generation.


## 2. Install/import what this backend needs

- **Colab (huggingface):** installs `transformers`, `torch`, `accelerate`
- **Local (ollama):** just needs the `ollama` python package (talks to your
  already-running local Ollama server)


In [3]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate
else:
    %pip install -q ollama


In [4]:
import warnings
warnings.filterwarnings("ignore")

if BACKEND == "huggingface":
    from transformers import pipeline
    import torch

    HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # swap for Qwen2.5-0.5B-Instruct if slow / CPU-only
    device = 0 if torch.cuda.is_available() else -1
    print(f"Loading {HF_MODEL} on {'GPU' if device == 0 else 'CPU'} ...")
    generator = pipeline("text-generation", model=HF_MODEL, device=device)

    def local_chat(messages, max_new_tokens=256, temperature=0.7):
        """messages: list of {"role": "system"|"user"|"assistant", "content": str}"""
        output = generator(
            messages,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
        )
        return output[0]["generated_text"][-1]["content"]

else:
    import ollama

    OLLAMA_MODEL = "llama3.2:3b"

    def local_chat(messages, max_new_tokens=256, temperature=0.7):
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=messages,
            options={"num_predict": max_new_tokens, "temperature": temperature},
        )
        return response["message"]["content"]

print("local_chat() is ready.")


Loading Qwen/Qwen2.5-1.5B-Instruct on GPU ...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

local_chat() is ready.


## 3. First call

Same `local_chat()` function works no matter which backend was selected above.


In [5]:
response = local_chat([
    {"role": "system", "content": "You are a concise, helpful teaching assistant."},
    {"role": "user", "content": "In one sentence, what is an LLM agent?"},
])
print(response)


[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


An LLM (Large Language Model) agent is a type of artificial intelligence designed to understand and generate human-like text based on vast amounts of data.


## 4. Prompting patterns

### Zero-shot


In [6]:
print(local_chat([
    {"role": "user", "content": "Classify the sentiment as positive, negative, or neutral: "
                                 "'The course was confusing but the instructor was helpful.'"}
]))


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To classify the sentiment of the given sentence, we need to analyze the overall tone and implications.

1. "The course was confusing": This phrase indicates that the speaker found the subject matter difficult to understand.
2. "but the instructor was helpful": The presence of an instructor who is helpful counteracts the initial confusion about the content being taught.

Considering these points together:
- The sentence starts with a negative aspect ("confusing") which suggests dissatisfaction with the learning experience.
- However, it ends on a more positive note ("helpful"), suggesting that despite the difficulty in understanding the material, there were beneficial aspects (e.g., guidance from the instructor).

Overall, while the primary impression is one of confusion due to the challenging nature of the subject matter, the presence of the helpful instructor mitigates this negativity somewhat.

**Sentiment Classification: Neutral**

This classification reflects the mixed nature of th

### Few-shot

In [7]:
few_shot_prompt = """Classify the sentiment as positive, negative, or neutral.

Text: "I loved this!"
Sentiment: positive

Text: "This was a waste of time."
Sentiment: negative

Text: "It was fine, nothing special."
Sentiment: neutral

Text: "The course was confusing but the instructor was helpful."
Sentiment:"""

print(local_chat([{"role": "user", "content": few_shot_prompt}], max_new_tokens=10))


[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


positive


### Chain-of-thought

In [8]:
cot_prompt = (
    "A store has 120 apples. It sells 45% of them in the morning and 30 more "
    "in the afternoon. How many apples are left? Think step by step, then give "
    "the final answer on its own line as 'Answer: <number>'."
)
print(local_chat([{"role": "user", "content": cot_prompt}], max_new_tokens=200))


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 1: Calculate the number of apples sold in the morning.
Number of apples sold in the morning = Total apples * Percentage sold in the morning
= 120 * 45%
= 120 * 0.45
= 54

Step 2: Subtract the number of apples sold in the morning from the total to find out how many are left after the morning sales.
Apples remaining after morning sales = Total apples - Apples sold in the morning
= 120 - 54
= 66

Step 3: Add the additional apples sold in the afternoon to find the total number of apples left.
Total apples left = Apples remaining after morning sales + Additional apples sold in the afternoon
= 66 + 30
= 96

Therefore, the final answer is Answer: 96.


## 5. Exercise

1. Re-run the sentiment classification few-shot prompt, but change the
   examples to a domain you care about (e.g. classifying support tickets by
   urgency instead of sentiment).
2. Try the same chain-of-thought math question with `temperature=0` vs
   `temperature=1.0` a few times — how much does the answer vary?
3. **(Colab users)** swap `HF_MODEL` for `Qwen/Qwen2.5-0.5B-Instruct` and
   compare speed and quality.
   **(Local users)** swap `OLLAMA_MODEL` for `qwen2.5:3b` and compare.

## Where this goes next
`lab2_structured_output.ipynb` — getting the model to reliably return JSON
instead of free text, which every later week's agent frameworks depend on.
